# Step 1：投影是否解释了 RA-ABDC

按 **问题 → 方法与消融 → 配对实验 → 结果 → 局限** 阅读已经保存的结果。规划、导航和评价都在 `src/udc/`，本笔记只读取轨迹和汇总。

## 问题

固定 Guide 数量时，Mass CVT 会把站点拉向需求质量高的弧段，从而让最大弧间隙超过规划上限。上一轮 RA-ABDC 的浓度始终是 1，没有回退。需要分开的是：可行性来自间隙投影，还是来自浓度调节。

人员权重即使全是 1，Gaussian 曲线密度也不平。评价必须使用这份原始密度，不能改用 $\rho^a$。

## 方法与消融

| 方法 | 规划密度 | 间隙投影 | 浓度调节 |
|---|---|---|---|
| Equal Arc | 均匀 | 无 | 无 |
| Mass CVT | 原始 $\rho$ | 无 | 无 |
| `projected_mass` | 原始 $\rho$ | 有 | 关闭 |
| RA-ABDC | $\rho^a$ | 有 | $a$ 减半，最后可记录 Equal Arc |

`projected_mass` 就是 RA-ABDC 的第一次尝试。只有投影后的欧氏间距失败时，两者才会分叉。导航和安全层相同。

## 配对实验

圆形与椭圆，50 人，均匀／弱热点（×2）／强热点（×5）。数量由 seed 0–4 的部署曲线决定：圆形 14／17／20，椭圆 16／18／22。四种方法，开发 seed 0–4，共 360 个任务。同一单元格共享人员、需求、边界和 Guide 初态。

```powershell
.\.venv\Scripts\python.exe -m udc.experiments.comparison --spec experiments/step1_projection_ablation.yaml --output runs/ablation/projection_matrix --plan-only --workers 8
.\.venv\Scripts\python.exe -m udc.experiments.comparison --spec experiments/step1_projection_ablation.yaml --output runs/ablation/projection_matrix --resume --workers 8
```

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / 'experiments' / 'step1_projection_ablation.yaml').exists() else Path.cwd().parent
matrix = ROOT / 'runs' / 'ablation' / 'projection_matrix'
rows = json.loads((matrix / 'comparison.json').read_text(encoding='utf-8'))
summary = json.loads((matrix / 'comparison_summary.json').read_text(encoding='utf-8'))
groups = defaultdict(set)
for row in rows:
    groups[(row['cell'], row['seed'])].add(row['input_snapshot_sha256'])
print('tasks', len(rows), 'successful', sum(row['success'] for row in rows))
print('pairs', len(groups), 'hash mismatches', sum(len(v) != 1 for v in groups.values()))
print('stages', {name: sum(row['stage_diagnosis'] == name for row in rows) for name in sorted({row['stage_diagnosis'] for row in rows})})
for method in ('equal_arc', 'mass_cvt', 'projected_mass', 'ra_abdc'):
    subset = [row for row in rows if row['method'] == method]
    print(method, sum(row['success'] for row in subset), '/', len(subset),
          'fallback', sum(bool(row.get('equal_arc_fallback')) for row in subset),
          'concentration', sorted({row.get('planned_concentration') for row in subset}))
display(Image(filename=str(matrix / 'comparison_summary.png')))
display(Image(filename=str(matrix / 'figures' / 'projected_mass_ellipse_strong_near_seed0.png')))
display(Image(filename=str(matrix / 'figures' / 'equal_arc_ellipse_strong_near_seed0.png')))

## 结果

360 个任务里 332 个成功，28 个是 `no_plan_found`，全部来自贴近数量下界的 Mass CVT。投影把这 28 个都变成通过验收的轨迹，并且没有破坏其余 62 个已经成功的 Mass CVT。没有指派失败或执行失败。90 对目标里，`projected_mass` 与 RA-ABDC 完全相同；浓度没有离开 1，回退次数为 0。

被挽救的 28 个任务上，原始密度的加权代价约为：无约束 Mass CVT 0.277，投影后 0.283，Equal Arc 0.288。相对 Equal Arc，投影留下约 44% 的代价改进。圆形最近人员窗口上的浓度从约 1.23 降到约 1.04，超出等弧长的集中约留下 16%。椭圆上这个窗口是空的，因为热点人员不是曲线的最近点；不能用该浓度描述椭圆。

## 局限

- 五个开发 seed 只支持机制诊断，不支持一般优劣结论。
- 浓度调节和 Equal Arc 回退没有被这些凸曲线触发，不能声称已经检验过。
- `ceil(L/H_p)` 只在“目标必须落在这条曲线上、弦长不超过弧长”时是必要条件。它不证明可行，搜索失败也不能升级成严格不可行。
- 4860 任务因子表、正式 seed 5000–5029、Step 2／3 都没有做。
- 在本轮几何上，完整 RA-ABDC 可以视为 `projected_mass`。没有证据支持再增加一层自适应机制，也不需要为此扩大到完整因子表。